# TabTransformer on 2xT4: multi-GPU ensembles with masaMLP

**TabTransformer** (Huang et al. 2020, [arXiv:2012.06678](https://arxiv.org/abs/2012.06678)) runs a transformer over the *categorical* features only, turning their embeddings into contextual embeddings; numeric features bypass attention and join at the MLP head. It shines on datasets where the signal lives in categorical columns — like the classic Adult census table used here.

This notebook uses [masaMLP](https://github.com/Matapanino/masamlp) — a PyTorch tabular deep-learning
library with sklearn-compatible estimators where **sample weights, custom
objectives, custom metrics, and early stopping on any metric** are
first-class.

**What this notebook shows.** masaMLP 0.3.0 trains seed ensembles
(`n_ens=k`) on **all detected GPUs at once**: with `device="auto"` and more
than one CUDA device, ensemble members are distributed round-robin across
the GPUs and trained concurrently, one worker thread per device. Kaggle's
free **GPU T4 x2** accelerator is exactly that setup, so below we fit the
same 4-member TabTransformer ensemble twice — pinned to one T4, then
sharded across both — and compare wall time and predictions.

> Accelerator: **Settings -> Accelerator -> "GPU T4 x2"**, and Internet ON
> (for `pip install` and the dataset download).

In [ ]:
%pip install -q masamlp==0.3.0

In [ ]:
import time

import numpy as np
import torch

import masamlp

N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"masamlp {masamlp.__version__} | torch {torch.__version__}")
print(f"CUDA devices: {N_GPUS}")
for i in range(N_GPUS):
    print(f"  cuda:{i} - {torch.cuda.get_device_name(i)}")
if N_GPUS < 2:
    print("\nNOTE: fewer than 2 GPUs detected. Choose Settings -> Accelerator ->")
    print("'GPU T4 x2' to see the multi-GPU sharding this notebook demonstrates.")

## Data: Adult census income

OpenML `adult` (~49k rows, 8 categorical + 6 numeric columns, binary target). masaMLP's built-in preprocessing detects the categorical dtypes automatically (`categorical_features="auto"`) and handles the missing values, so the DataFrame goes straight into `fit`.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

adult = fetch_openml("adult", version=2, as_frame=True)
X, y = adult.data, adult.target  # 48842 rows: 6 numeric + 8 categorical
n_cat = int((X.dtypes == "category").sum())
print(f"{n_cat} categorical + {X.shape[1] - n_cat} numeric columns")

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=0)
print(f"train {X_train.shape}  valid {X_valid.shape}  test {X_test.shape}")

## Fit the same ensemble twice

`fit_timed` trains a 4-member seed ensemble (TabTransformer, members
seeded `random_state + i`, predictions averaged on the probability scale).
The only thing we change between the two runs is `device`:

- `device="cuda:0"` pins everything to one GPU (this is also the opt-out
  when you *don't* want sharding);
- `device="auto"` detects both T4s and trains two members on each,
  concurrently.

Early stopping monitors log-loss — a probability-quality metric is a much
steadier stopping signal than a discrete metric like accuracy.

In [ ]:
from masamlp import MasaClassifier


def fit_timed(device: str):
    model = MasaClassifier(
        model="tab_transformer",
        n_epochs=40,
        early_stopping_rounds=5,
        eval_metric="logloss",
        n_ens=4,
        device=device,
        random_state=0,
    )
    start = time.perf_counter()
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)])
    return model, time.perf_counter() - start


single_model, single_sec = fit_timed("cuda:0" if N_GPUS else "cpu")
print(f"single-device fit: {single_sec:.1f}s  "
      f"(best epoch {single_model.best_iteration_})")

In [ ]:
sharded_model, sharded_sec = fit_timed("auto")
print(f"sharded fit:       {sharded_sec:.1f}s  "
      f"(best epoch {sharded_model.best_iteration_})")
print("member devices:",
      {i: str(next(m.parameters()).device)
       for i, m in enumerate(sharded_model.models_)})

## Results

Wall time, validation quality, and how far apart the two runs' predictions
are. Weight initialization is identical (it is seeded on the main thread in
both modes); training on a different GPU topology changes the RNG streams
and float accumulation, so predictions agree closely but not bitwise.

In [ ]:
from sklearn.metrics import log_loss

p_single = single_model.predict_proba(X_valid)
p_sharded = sharded_model.predict_proba(X_valid)
ll_single = log_loss(y_valid, p_single, labels=single_model.classes_)
ll_sharded = log_loss(y_valid, p_sharded, labels=sharded_model.classes_)

print(f"{'run':<12} {'fit wall time':>14} {'valid logloss':>14}")
print(f"{'single GPU':<12} {single_sec:>13.1f}s {ll_single:>14.5f}")
print(f"{'sharded':<12} {sharded_sec:>13.1f}s {ll_sharded:>14.5f}")
if N_GPUS >= 2:
    print(f"\nsharded speedup: x{single_sec / sharded_sec:.2f} on {N_GPUS} GPUs")
    print(f"max |prediction diff|: {np.abs(p_single - p_sharded).max():.2e}")
else:
    print("\nSingle GPU detected: device='auto' cannot shard, so both runs")
    print("took the same code path. Re-run with 'GPU T4 x2' for the speedup.")

In [ ]:
from sklearn.metrics import accuracy_score

pred = sharded_model.predict(X_test)
print(f"held-out test accuracy ({len(X_test)} rows): "
      f"{accuracy_score(y_test, pred):.4f}")

## Notes

- **Opting out / picking GPUs**: `device="cuda:0"` (any explicit index)
  disables sharding. `device="auto"` shards only when `n_ens > 1` and more
  than one CUDA device is visible.
- **Reproducibility**: same seed + same GPU topology gives the same result.
  Sharded vs single-GPU runs agree closely (see the diff above) but are not
  guaranteed bitwise-equal.
- With **no categorical columns** TabTransformer degenerates to its MLP head — pick a dataset like this one, where the transformer has tokens to attend over.
- On a single GPU it also supports `ens_mode="vectorized"` (LayerNorm-only architecture); loop mode is what shards across GPUs.
- Library: [masaMLP](https://github.com/Matapanino/masamlp) — models, objectives/metrics plugins, and the
  multi-GPU details are documented in `docs/` (see `docs/devices.md`).